# 02 — Agent evaluation (oracle mode)

This notebook runs **Qwen3.5-9B** as a tool-using agent on 50 benchmark
questions. The agent has three tools:

1. `segment_organ(target)` — voxel count + bounding box
2. `measure(target, measurement_type)` — volume, HU, diameter, lesion count, ...
3. `lookup_medical_knowledge(query)` — clinical guideline retrieval

In **oracle mode**, the tools return GT-mask-derived values from the shipped
`tool_cache/benchmark_oracle_tool_cache.json` — no live segmentation needed.

We use a vLLM OpenAI-compatible server as the agent backbone.

Expected wall time: ~5 min for 50 samples on a 1×A6000 (after server warmup).


## 1. Install


In [ ]:
!pip install -q deeptumorvqa[openai]
!pip install -q vllm   # required to host the agent backbone


## 2. Start a vLLM server (in a separate terminal)

```bash
python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen3.5-9B \
    --port 8877 \
    --max-model-len 8192 \
    --gpu-memory-utilization 0.85 \
    --dtype bfloat16 \
    --enable-auto-tool-choice \
    --tool-call-parser hermes
```

Wait until you see `Application startup complete` before continuing.


## 3. Run the agent evaluator


In [ ]:
!deeptumorvqa-eval \
    --model Qwen/Qwen3.5-9B \
    --backend openai \
    --api-base http://localhost:8877/v1 \
    --mode agent --agent-mode oracle --format mc \
    --limit 50 \
    --output results/qwen35_agent_oracle.json \
    --label "Qwen3.5-9B (oracle, smoke)"


## 4. Inspect a tool-use trajectory

Each sample's `trajectory` records every tool call the agent made,
along with the cached result and the final answer.


In [ ]:
import json
data = json.load(open("results/qwen35_agent_oracle.json"))
print("Overall:", data["metrics"]["overall"])
print()

# Show one full trajectory
sample = next(r for r in data["results"] if r.get("trajectory"))
print(f"Question: {sample['raw_output'][:80]}...")
print(f"Final answer: {sample['raw_output']}  (correct: {sample['correct_option']})")
print(f"\nTrajectory ({len(sample['trajectory'])} steps):")
for step in sample["trajectory"]:
    if "tool" in step:
        print(f"  step {step['step']}: {step['tool']}({step['arguments']})")
        print(f"           -> {json.dumps(step['result'])[:120]}")
    else:
        print(f"  step {step['step']}: FINAL = {step.get('final_answer','')[:60]}")


## 5. Other agent modes

- `--agent-mode predicted`: same tools, but values come from
  `benchmark_totalsegmentator_cache.json` (auto-segmentation, noisier)
- `--agent-mode vision`: replaces measurement tools with `crop_organ`,
  forcing the model to visually estimate measurements

Paper numbers for full 10K benchmark (Table 2):

| Mode | Qwen3.5-9B | Meissa-4B | Meissa SFT |
|---|---|---|---|
| oracle    | 48.5% | 46.0% | 63.8% |
| predicted | 43.9% | —     | 60.9% |
| vision    | 40.3% | 31.5% | —     |
